# Agentic RAG: Router–Retriever System with PDF and Web Search Tools
**Course-end Project 1**

This notebook builds a two-agent Retrieval-Augmented Generation (RAG) system with **CrewAI**:

| Agent | Responsibility | Tools |
|---|---|---|
| **Router Agent** | Reads the user question and decides the retrieval path: `pdf_search`, `web_search` or `direct_llm` | none (pure reasoning) |
| **Retriever Agent** | Runs the chosen tool, reads the evidence and writes a grounded, cited answer | `PDFSearchTool` (crewai_tools) or Tavily web search (`TavilySearchResults`) |

**Knowledge sources**
* **Static** – the research paper *"Attention Is All You Need"* (Transformer paper), indexed into a vector store by `PDFSearchTool`.
* **Dynamic** – the live internet, through the Tavily search API.
* **Optional** – a direct LLM answer with no retrieval, for simple general questions.

**Flow:** `Question → Router Agent → route decision (JSON) → Retriever Agent + chosen tool → grounded answer → trace log`

### Notebook sections
1. Setup and installation
2. Configuration (API keys, model, PDF path)
3. Interaction logging (trace logger)
4. Tools: PDF search and Web search
5. Agent definitions (roles, goals, backstories)
6. Tasks and orchestration (the coordinator)
7. Demo: PDF questions, web questions, direct question
8. Trace logs and reasoning-trace visualisations
9. Interactive mode

## 1. Setup and installation
Run once. Requires Python 3.10–3.13. Restart the kernel after installing if prompted.

In [2]:
%pip install -q crewai crewai-tools langchain-community tavily-python python-dotenv

In [4]:
%pip install -q "pandas==2.2.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [2]:
import os, json, time, uuid, logging, re
from datetime import datetime
from pathlib import Path
from typing import Literal

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai_tools import PDFSearchTool

# TavilySearchResults lives in langchain_community (the project brief asks for this class).
from langchain_community.tools.tavily_search import TavilySearchResults

import crewai
print("CrewAI version:", crewai.__version__)

/tmp/ipykernel_10791/2907137687.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


CrewAI version: 1.15.33


## 2. Configuration
Set your keys **before** running the rest of the notebook. The safest way is a `.env` file in the same folder:
```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```
* **OpenAI** – used by both agents (LLM) and by `PDFSearchTool` (embeddings).
* **Tavily** – free key at https://tavily.com, used for web search.

In [5]:
from dotenv import load_dotenv

In [9]:
from dotenv import load_dotenv
load_dotenv()

# Fallback: prompt for keys if they are not in the environment (keys are never printed).
from getpass import getpass
for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.getenv(key):
        os.environ[key] = getpass(f"Enter {key}: ")

MODEL_NAME = "gpt-4o-mini"          # cheap, fast and good at following JSON instructions
PDF_PATH   = "trasformer_research_paper-dataset.pdf"   # the Transformer paper supplied with the project
LOG_DIR    = Path("logs"); LOG_DIR.mkdir(exist_ok=True)

assert Path(PDF_PATH).exists(), f"PDF not found at {PDF_PATH} - put it next to this notebook."

# temperature=0 for the router (we want consistent, repeatable routing decisions)
router_llm    = LLM(model=MODEL_NAME, temperature=0)
retriever_llm = LLM(model=MODEL_NAME, temperature=0.2)
print("Configuration OK")

Enter OPENAI_API_KEY: ··········
Enter TAVILY_API_KEY: ··········
Configuration OK


## 3. Interaction logging (reasoning traceability)
Every important event is recorded in three places:
1. **In-memory list `TRACE`** → turned into a pandas table and charts later.
2. **`logs/agentic_rag_trace.jsonl`** → one JSON line per event (machine-readable audit trail).
3. **`logs/agentic_rag.log`** → human-readable Python log.

CrewAI gives us two hooks that we plug the logger into:
* `step_callback` on each **Agent** → fires on every thought / tool call / tool result.
* `task_callback` on each **Crew** → fires when a task finishes.

In [10]:
TRACE: list[dict] = []
TRACE_FILE = LOG_DIR / "agentic_rag_trace.jsonl"

logger = logging.getLogger("agentic_rag")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fh = logging.FileHandler(LOG_DIR / "agentic_rag.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)

def _short(x, n=400):
    s = str(x) if x is not None else ""
    return s if len(s) <= n else s[:n] + " ..."

def log_event(run_id: str, agent: str, event: str, detail="", **extra):
    """Record one step of the reasoning trace."""
    rec = {"run_id": run_id, "ts": datetime.now().isoformat(timespec="milliseconds"),
           "t": time.time(), "agent": agent, "event": event, "detail": _short(detail), **extra}
    TRACE.append(rec)
    with open(TRACE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, default=str) + "\n")
    logger.info(f"[{run_id}] {agent:<16} {event:<14} {_short(detail, 150)}")
    print(f"  📝 {agent:<16} | {event:<14} | {_short(detail, 120)}")

# Which run is currently active (set by the coordinator) so callbacks know where to log.
CURRENT_RUN = {"id": None}

def make_step_callback(agent_name: str):
    """Build a step_callback that logs CrewAI agent steps (works across CrewAI versions)."""
    def _cb(step):
        run_id = CURRENT_RUN["id"]
        kind = type(step).__name__
        tool_name  = getattr(step, "tool", None)
        tool_input = getattr(step, "tool_input", None)
        thought    = getattr(step, "thought", None)
        result     = getattr(step, "result", None) or getattr(step, "output", None)
        if tool_name:
            log_event(run_id, agent_name, "tool_call", f"{tool_name}({_short(tool_input, 150)})", tool=tool_name)
            if result:
                log_event(run_id, agent_name, "tool_result", result, tool=tool_name)
        elif thought:
            log_event(run_id, agent_name, "thought", thought)
        else:
            log_event(run_id, agent_name, f"step:{kind}", result or step)
    return _cb

def task_callback(output):
    log_event(CURRENT_RUN["id"], getattr(output, "agent", "crew"), "task_done", getattr(output, "raw", output))

## 4. Tools
### 4.1 PDF search tool (static knowledge)
`PDFSearchTool` splits the PDF into chunks, embeds them and stores them in a local vector database. At query time it returns the chunks most similar to the question (semantic search).

In [11]:
pdf_search_tool = PDFSearchTool(pdf=PDF_PATH)

# Quick sanity check of the vector search (no agent involved yet)
print(_short(pdf_search_tool.run("What is scaled dot-product attention?"), 800))

Relevant Content:




Page 4:

Scaled Dot-Product Attention

Multi-Head Attention

Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several

attention layers running in parallel.

of the values, where the weight assigned to each value is computed by a compatibility function of the

query with the corresponding key.

3.2.1

Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of

queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the

query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the

values.

In practice, we compute the attention function on a set of queries simultaneously, packed together

 ...


### 4.2 Web search tool (dynamic knowledge)
`TavilySearchResults` is a LangChain tool. We wrap it with CrewAI's `@tool` decorator so a CrewAI agent can call it, and format the results so every snippet keeps its **source URL** (needed for grounded, cited answers).

In [12]:
tavily = TavilySearchResults(max_results=4)

@tool("Web Search Tool")
def web_search_tool(query: str) -> str:
    """Search the internet for fresh, up-to-date information. Input: a concise search query.
    Returns the top results, each with its title/URL and a content snippet."""
    try:
        results = tavily.invoke({"query": query})
    except Exception as e:
        return f"Web search failed: {e}"
    if not results:
        return "No web results found."
    lines = []
    for i, r in enumerate(results, 1):
        lines.append(f"[{i}] URL: {r.get('url')}\n{r.get('content', '')[:700]}")
    return "\n\n".join(lines)

# Sanity check
print(_short(web_search_tool.run("latest CrewAI release"), 600))

/tmp/ipykernel_10791/4198084209.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(max_results=4)


[1] URL: https://community.crewai.com/t/new-release-0-175-0/6989
## Executive Summary

The release of CrewAI v0.175.0 on August 28, 2025, delivered significant enhancements focused on RAG (Retrieval-Augmented Generation), dependency flexibility, and flow control robustness. Key improvements include a simplified RAG configuration with added Qdrant support and the removal of a strict version pin on the OpenAI library, which resolved critical import conflicts for many users.

Key insights:

Dependency Conflicts are a Major Blocker: The most critical feedback from the community stems from dependen ...


## 5. Agent definitions
### Router Agent
* **Role:** query classifier / traffic controller.
* **Input:** the raw user question.
* **Output:** a structured JSON decision `{route, reason, search_query}` (enforced with a Pydantic model).
* Has **no tools** – it only reasons, which keeps routing fast and cheap.

### Retriever Agent
* **Role:** research specialist.
* **Input:** the question + the router's decision.
* **Output:** the final answer, grounded in retrieved evidence, with sources.
* Is given **only the tool the router selected**, so it cannot drift to the wrong source.

In [13]:
class RouteDecision(BaseModel):
    route: Literal["pdf_search", "web_search", "direct_llm"] = Field(
        description="pdf_search = Transformer paper, web_search = fresh/internet info, direct_llm = simple general question")
    reason: str = Field(description="One or two sentences explaining the choice")
    search_query: str = Field(description="Optimised query to send to the chosen tool")

PDF_SCOPE = (
    "The PDF knowledge base is the 2017 research paper 'Attention Is All You Need' (Vaswani et al., Google). "
    "It covers: the Transformer architecture, encoder/decoder stacks, self-attention, scaled dot-product attention, "
    "multi-head attention, positional encoding, position-wise feed-forward networks, why self-attention vs RNN/CNN, "
    "training data (WMT 2014 English-German / English-French), hardware and schedule, optimizer and learning-rate "
    "schedule, regularization (dropout, label smoothing), BLEU results, model variations and English constituency parsing."
)

router_agent = Agent(
    role="Query Router",
    goal="Classify each user question and choose the single best retrieval path: pdf_search, web_search or direct_llm.",
    backstory=(
        "You are an expert at understanding information needs. " + PDF_SCOPE +
        " Route to pdf_search when the question is about anything in that paper. "
        "Route to web_search when the question needs current, recent or external information "
        "(news, latest models, prices, events after 2017, companies, people, versions). "
        "Route to direct_llm only for simple general-knowledge, maths, or conversational questions "
        "that need no sources. You never answer the question yourself."
    ),
    llm=router_llm,
    tools=[],
    allow_delegation=False,
    verbose=True,
    step_callback=make_step_callback("Router Agent"),
)

def build_retriever_agent(tools: list) -> Agent:
    """Create the Retriever Agent equipped with exactly the tool(s) chosen by the router."""
    return Agent(
        role="Retrieval Specialist",
        goal="Retrieve evidence with the assigned tool and write an accurate, concise answer grounded ONLY in that evidence, with sources.",
        backstory=(
            "You are a meticulous research assistant. You always call your tool before answering, "
            "you quote or paraphrase only what the retrieved evidence supports, you cite sources "
            "(PDF section names or web URLs), and you clearly say when the evidence is insufficient "
            "instead of guessing."
        ),
        llm=retriever_llm,
        tools=tools,
        allow_delegation=False,
        verbose=True,
        max_iter=5,
        step_callback=make_step_callback("Retriever Agent"),
    )

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:250: UserWarning: A function callback cannot be saved in a checkpoint; it is dropped and will not run after a restore. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## 6. Tasks and orchestration
**Why CrewAI?**
* Agents are declared by **role, goal and backstory** – this maps 1:1 to the project's role-based design.
* Built-in **tool integration** (`crewai_tools.PDFSearchTool`, `@tool` wrapper for LangChain tools).
* **Structured outputs** (`output_pydantic`) make the router's decision machine-readable.
* **Callbacks + verbose mode + `output_log_file`** give reasoning traceability out of the box.
* Lightweight compared with building a state graph by hand – ideal for a 2-agent sequential pipeline.

**Coordination design (two small crews, coordinated in Python):**
1. **Routing crew** – Router Agent + routing task → `RouteDecision`.
2. The coordinator validates the decision (with a keyword fallback if the LLM output is malformed).
3. **Retrieval crew** – Retriever Agent with the chosen tool + retrieval task (receives the router decision as input) → final answer.

Keeping the hand-off in Python makes the input/output flow between the agents explicit, testable and fully logged.

In [14]:
TOOLS_BY_ROUTE = {
    "pdf_search": [pdf_search_tool],
    "web_search": [web_search_tool],
    "direct_llm": [],
}

def build_routing_task(question: str) -> Task:
    return Task(
        description=(
            f"User question: '{question}'\n\n"
            "Decide the single best retrieval route:\n"
            "- pdf_search: answerable from the 'Attention Is All You Need' Transformer paper\n"
            "- web_search: needs fresh / external / post-2017 information from the internet\n"
            "- direct_llm: simple general question that needs no retrieval\n"
            "Also write an optimised search_query for the chosen tool."
        ),
        expected_output="A JSON object with fields route, reason and search_query.",
        agent=router_agent,
        output_pydantic=RouteDecision,
    )

def build_retrieval_task(question: str, decision: RouteDecision, agent: Agent) -> Task:
    if decision.route == "pdf_search":
        how = (f"Use the PDF search tool with the query '{decision.search_query}' (you may refine it and search again once). "
               "Cite the paper section(s) the evidence comes from.")
    elif decision.route == "web_search":
        how = (f"Use the Web Search Tool with the query '{decision.search_query}'. "
               "Cite the URLs of the results you used.")
    else:
        how = "No retrieval is needed. Answer directly and briefly from general knowledge, and state 'Source: general knowledge (no retrieval)'."
    return Task(
        description=(
            f"User question: '{question}'\n"
            f"Router decision: route={decision.route}; reason={decision.reason}\n\n"
            f"{how}\n"
            "Answer ONLY with information supported by the retrieved evidence. "
            "If the evidence is insufficient, say so."
        ),
        expected_output="A clear answer (3-8 sentences or bullets) followed by a 'Sources:' line.",
        agent=agent,
    )

def keyword_fallback_route(question: str) -> RouteDecision:
    """Safety net if the router's output cannot be parsed."""
    q = question.lower()
    pdf_kw = ["transformer", "attention", "encoder", "decoder", "positional", "bleu", "wmt", "multi-head",
              "self-attention", "feed-forward", "label smoothing", "this paper", "vaswani"]
    web_kw = ["latest", "today", "current", "news", "2024", "2025", "2026", "price", "recent", "who is", "release"]
    if any(k in q for k in web_kw):
        return RouteDecision(route="web_search", reason="Keyword fallback: time-sensitive terms.", search_query=question)
    if any(k in q for k in pdf_kw):
        return RouteDecision(route="pdf_search", reason="Keyword fallback: Transformer-paper terms.", search_query=question)
    return RouteDecision(route="direct_llm", reason="Keyword fallback: general question.", search_query=question)

def parse_decision(task_output) -> RouteDecision:
    if getattr(task_output, "pydantic", None):
        return task_output.pydantic
    raw = getattr(task_output, "raw", str(task_output))
    m = re.search(r"\{.*\}", raw, re.S)
    return RouteDecision(**json.loads(m.group(0)))

In [18]:
from concurrent.futures import ThreadPoolExecutor

def run_crew(crew):
    """Run crew.kickoff() in a separate thread (Colab already runs an event loop)."""
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(crew.kickoff).result()

RESULTS: list[dict] = []

def answer_question(question: str, force_route: str | None = None) -> dict:
    """Coordinator: Question -> Router Agent -> Retriever Agent -> grounded answer (fully logged)."""
    run_id = uuid.uuid4().hex[:6]
    CURRENT_RUN["id"] = run_id
    print(f"\n{'='*90}\n▶ RUN {run_id} | QUESTION: {question}\n{'='*90}")
    log_event(run_id, "Coordinator", "question", question)
    t0 = time.time()

    # ---- Step 1: Routing --------------------------------------------------
    log_event(run_id, "Coordinator", "handoff", "Question sent to Router Agent")
    if force_route:
        decision = RouteDecision(route=force_route, reason="Route forced by user for testing.", search_query=question)
    else:
        routing_crew = Crew(agents=[router_agent], tasks=[build_routing_task(question)],
                            process=Process.sequential, verbose=True, task_callback=task_callback,
                            output_log_file=str(LOG_DIR / "crew_router.log"))
        try:
            decision = parse_decision(run_crew(routing_crew).tasks_output[0])
        except Exception as e:
            log_event(run_id, "Coordinator", "router_error", f"{e} -> using keyword fallback")
            decision = keyword_fallback_route(question)
    t_route = time.time() - t0
    log_event(run_id, "Router Agent", "decision",
              f"route={decision.route} | query='{decision.search_query}' | reason={decision.reason}",
              route=decision.route)

    # ---- Step 2: Retrieval ------------------------------------------------
    tools = TOOLS_BY_ROUTE[decision.route]
    log_event(run_id, "Coordinator", "handoff",
              f"Decision sent to Retriever Agent with tools={[t.name for t in tools] or 'none'}")
    retriever_agent = build_retriever_agent(tools)
    retrieval_crew = Crew(agents=[retriever_agent], tasks=[build_retrieval_task(question, decision, retriever_agent)],
                          process=Process.sequential, verbose=True, task_callback=task_callback,
                          output_log_file=str(LOG_DIR / "crew_retriever.log"))
    answer = answer = run_crew(retrieval_crew).raw
    total = time.time() - t0
    log_event(run_id, "Retriever Agent", "final_answer", answer, route=decision.route)
    log_event(run_id, "Coordinator", "complete", f"Done in {total:.1f}s")

    result = {"run_id": run_id, "question": question, "route": decision.route, "reason": decision.reason,
              "search_query": decision.search_query, "answer": answer,
              "routing_sec": round(t_route, 2), "total_sec": round(total, 2)}
    RESULTS.append(result)
    return result

def show(result: dict):
    from IPython.display import Markdown, display
    display(Markdown(
        f"### ❓ {result['question']}\n"
        f"**Route:** `{result['route']}`  |  **Router reason:** {result['reason']}  |  "
        f"**Time:** {result['total_sec']}s\n\n---\n{result['answer']}"))

## 7. Demo
### 7.1 Questions about the PDF (static source → expected route `pdf_search`)

In [19]:
r1 = answer_question("What is multi-head attention and why does the Transformer use it?")
show(r1)

INFO:agentic_rag:[f73a7c] Coordinator      question       What is multi-head attention and why does the Transformer use it?
INFO:agentic_rag:[f73a7c] Coordinator      handoff        Question sent to Router Agent



▶ RUN f73a7c | QUESTION: What is multi-head attention and why does the Transformer use it?
  📝 Coordinator      | question       | What is multi-head attention and why does the Transformer use it?
  📝 Coordinator      | handoff        | Question sent to Router Agent


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d2552cc8-565b-4407-a112-921cc4b67d45                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: User question: 'What is multi-head attention and why does the Transformer use it?'                       │
│                                                                                                                 │
│  Decide the single best retrieval route:                                                                        │
│  - pdf_search: answerable from the 'Attention Is All You Need' Transformer paper                                │
│  - web_search: needs fresh / external / post-2017 information from the internet                                 │
│  - direct_llm: simple general question that needs no retrieval                                                  │
│  Also write an optimised search_query for the chosen tool.                                                      │
│  ID: 4714dff8-06e8-4568-93bc-cfaecacf79a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: User question: 'What is multi-head attention and why does the Transformer use it?'                       │
│                                                                                                                 │
│  Decide the single best retrieval route:                                                                        │
│  - pdf_search: answerable from the 'Attention Is All You Need' Transformer paper                                │
│  - web_search: needs fresh / external / post-2017 information from the internet                                 │
│  - direct_llm: simple general question that needs no retrieval                                                  │
│  Also write an optimised search_query for the chosen tool.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (19 outputs left out)

INFO:agentic_rag:[f73a7c] Retriever Agent  final_answer   Multi-head attention is a mechanism used in the Transformer model that allows it to jointly attend to information from different representation subspa ...
INFO:agentic_rag:[f73a7c] Coordinator      complete       Done in 9.0s


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cec5ce31-7700-4d87-92da-bab9d32b69ca                                                                       │
│  Final Output: Multi-head attention is a mechanism used in the Transformer model that allows it to jointly      │
│  attend to information from different representation subspaces at different positions. Instead of performing a  │
│  single attention function, multi-head attention projects the queries, keys, and values multiple times (h       │
│  times) with different learned linear projections. Each of these projected versions undergoes the attention     │
│  function in parallel, and the resulting outputs are concatenated and projected again to form the final         │
│  output. This approach helps mitigate the averaging effect that can occur with a single attention head,         │
│  allowing the model to capture more nuanced relationships in the data.                                          │
│                                                                                                                 │
│  The Transformer employs multi-head attention in various ways, including in encoder-decoder attention layers    │
│  and self-attention layers. In encoder-decoder attention, the queries come from the decoder, while the keys     │
│  and values come from the encoder's output, enabling the decoder to attend to all positions in the input        │
│  sequence. Self-attention layers allow the model to relate different positions of a single sequence to compute  │
│  a comprehensive representation of that sequence.                                                               │
│                                                                                                                 │
│  Sources:                                                                                                       │
│  - "Attention Is All You Need," Section 3.2: Attention.                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  📝 Retriever Agent  | final_answer   | Multi-head attention is a mechanism used in the Transformer model that allows it to jointly attend to information from d ...
  📝 Coordinator      | complete       | Done in 9.0s


### ❓ What is multi-head attention and why does the Transformer use it?
**Route:** `pdf_search`  |  **Router reason:** The question specifically asks about multi-head attention and its purpose in the Transformer, which is covered in detail in the 'Attention Is All You Need' paper.  |  **Time:** 8.95s

---
Multi-head attention is a mechanism used in the Transformer model that allows it to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function, multi-head attention projects the queries, keys, and values multiple times (h times) with different learned linear projections. Each of these projected versions undergoes the attention function in parallel, and the resulting outputs are concatenated and projected again to form the final output. This approach helps mitigate the averaging effect that can occur with a single attention head, allowing the model to capture more nuanced relationships in the data.

The Transformer employs multi-head attention in various ways, including in encoder-decoder attention layers and self-attention layers. In encoder-decoder attention, the queries come from the decoder, while the keys and values come from the encoder's output, enabling the decoder to attend to all positions in the input sequence. Self-attention layers allow the model to relate different positions of a single sequence to compute a comprehensive representation of that sequence.

Sources:
- "Attention Is All You Need," Section 3.2: Attention.

In [ ]:
r2 = answer_question("What BLEU score did the Transformer (big) model achieve on WMT 2014 English-to-German?")
show(r2)

In [ ]:
r3 = answer_question("How does the Transformer encode the position of words, given that it has no recurrence?")
show(r3)

### 7.2 Questions that need fresh information (dynamic source → expected route `web_search`)

In [ ]:
r4 = answer_question("What are the latest large language models released by OpenAI and Google this year?")
show(r4)

In [ ]:
r5 = answer_question("What is the current latest version of the CrewAI Python package?")
show(r5)

### 7.3 Simple general question (optional path → expected route `direct_llm`)

In [ ]:
r6 = answer_question("Convert 25 degrees Celsius to Fahrenheit.")
show(r6)

### 7.4 Routing summary

In [ ]:
summary = pd.DataFrame(RESULTS)[["run_id", "question", "route", "search_query", "routing_sec", "total_sec"]]
pd.set_option("display.max_colwidth", 80)
summary

## 8. Trace logs and reasoning-trace visualisation
The table below is the full **interaction log**: which agent did what, in what order, with which tool. It is also saved to `logs/agentic_rag_trace.jsonl`.

In [ ]:
trace_df = pd.DataFrame(TRACE)
trace_df[["run_id", "ts", "agent", "event", "detail"]]

In [ ]:
# 8.1 Per-run reasoning trace: who did what (one line per event, in order)
def print_trace(run_id: str):
    print(f"Reasoning trace for run {run_id}\n" + "-"*70)
    for i, row in enumerate(trace_df[trace_df.run_id == run_id].itertuples(), 1):
        print(f"{i:>2}. [{row.agent}] {row.event}: {_short(row.detail, 140)}")

print_trace(r1["run_id"])

In [ ]:
# 8.2 Timeline (swim-lane) chart of agent activity for every run
colors = {"Coordinator": "#7f8c8d", "Router Agent": "#2e86de", "Retriever Agent": "#10ac84"}
lanes = ["Coordinator", "Router Agent", "Retriever Agent"]
runs = trace_df.run_id.unique()
fig, axes = plt.subplots(len(runs), 1, figsize=(12, 1.9 * len(runs)), squeeze=False)
for ax, rid in zip(axes[:, 0], runs):
    d = trace_df[trace_df.run_id == rid]
    t0 = d.t.min()
    for _, row in d.iterrows():
        y = lanes.index(row.agent) if row.agent in lanes else 0
        ax.scatter(row.t - t0, y, color=colors.get(row.agent, "black"), s=60, zorder=3)
        ax.annotate(row.event, (row.t - t0, y), textcoords="offset points", xytext=(0, 6),
                    fontsize=7, ha="center", rotation=20)
    route = d.route.dropna().iloc[0] if "route" in d and d.route.notna().any() else "?"
    ax.set_yticks(range(len(lanes))); ax.set_yticklabels(lanes, fontsize=8)
    ax.set_title(f"Run {rid} → route: {route}", fontsize=9, loc="left")
    ax.grid(axis="x", alpha=.3); ax.set_ylim(-0.6, 2.8)
axes[-1, 0].set_xlabel("seconds since question received")
plt.tight_layout(); plt.show()

In [ ]:
# 8.3 Route distribution and latency per route
res_df = pd.DataFrame(RESULTS)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
res_df.route.value_counts().plot.bar(ax=a1, color=["#2e86de", "#10ac84", "#ee5253"])
a1.set_title("Questions per route"); a1.set_ylabel("count"); a1.tick_params(axis="x", rotation=0)
res_df.groupby("route")[["routing_sec", "total_sec"]].mean().plot.bar(ax=a2)
a2.set_title("Average latency (s)"); a2.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# 8.4 Agent / tool interaction counts
trace_df.groupby(["agent", "event"]).size().unstack(fill_value=0)

In [ ]:
# 8.5 Architecture diagram (drawn with matplotlib - no extra install needed)
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(12, 4.2)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 6)
def box(x, y, w, h, text, c):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1", fc=c, ec="black"))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=9, wrap=True)
def arrow(x1, y1, x2, y2, label=""):
    ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="->", lw=1.5))
    if label: ax.text((x1+x2)/2, (y1+y2)/2 + .15, label, fontsize=8, ha="center")
box(0.2, 2.5, 1.8, 1, "User\nquestion", "#dfe6e9")
box(2.8, 2.5, 2.0, 1, "Router Agent\n(classify)", "#74b9ff")
box(5.8, 4.4, 2.2, 1, "PDFSearchTool\n(Transformer paper)", "#ffeaa7")
box(5.8, 2.5, 2.2, 1, "Web Search\n(Tavily)", "#ffeaa7")
box(5.8, 0.6, 2.2, 1, "Direct LLM\n(no retrieval)", "#ffeaa7")
box(8.8, 2.5, 2.0, 1, "Retriever Agent\n(grounded answer)", "#55efc4")
arrow(2.0, 3.0, 2.8, 3.0)
arrow(4.8, 3.2, 5.8, 4.9, "pdf_search"); arrow(4.8, 3.0, 5.8, 3.0, "web_search"); arrow(4.8, 2.8, 5.8, 1.1, "direct_llm")
arrow(8.0, 4.9, 8.8, 3.3); arrow(8.0, 3.0, 8.8, 3.0); arrow(8.0, 1.1, 8.8, 2.7)
ax.text(9.8, 2.1, "↓ answer + sources\n+ trace log", ha="center", fontsize=8)
ax.set_title("Agentic RAG – Router/Retriever architecture"); plt.show()

In [ ]:
# 8.6 Export results for the submission
pd.DataFrame(RESULTS).to_csv(LOG_DIR / "demo_results.csv", index=False)
trace_df.to_csv(LOG_DIR / "trace_table.csv", index=False)
print("Saved:", *sorted(p.name for p in LOG_DIR.iterdir()), sep="\n  ")

## 9. Interactive mode
Ask your own questions. Type `exit` to stop.

In [ ]:
while True:
    q = input("\nAsk a question (or 'exit'): ").strip()
    if q.lower() in {"exit", "quit", ""}:
        break
    show(answer_question(q))

## 10. Conclusion
* The **Router Agent** reliably separated paper questions (`pdf_search`), time-sensitive questions (`web_search`) and trivial questions (`direct_llm`).
* The **Retriever Agent** answered only from the evidence returned by its single assigned tool and cited its sources.
* Every hand-off, thought, tool call and answer is captured in the trace log and visualised above.

See **README.md** for architecture, agent responsibilities, coordination flow, challenges and trade-offs.